# schedule_paper_execution_model_difference_v1

目标：验证 `src/schedule` 的 paper execution pipeline 在读取 dashboard export 和 canonical data registry 后，能把 `model_difference_explanations` 产生的风控原因写入 schedule payload、详细通知、intent ledger 和 paper execution result。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_canonical_snapshot, publish_canonical_snapshot
from src.schedule import PaperExecutionPipelineConfig, run_paper_execution_once

EXPERIMENT_DIR = REPO_ROOT / "research/schedule_paper_execution_model_difference_v1"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

def write_json(path: Path, payload: dict | list) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

raw = pd.DataFrame(
    [
        {"date": "2026-01-02", "instrument": "SOXL", "close": 10.0, "volume": 100},
        {"date": "2026-01-05", "instrument": "SOXL", "close": 12.0, "volume": 120},
    ]
)
calendar = pd.DatetimeIndex(pd.to_datetime(["2026-01-02", "2026-01-05"]), name="date")
snapshot = build_canonical_snapshot(
    raw,
    name="schedule_paper_difference_daily",
    universe=("SOXL",),
    calendar=calendar,
    calendar_path="data/qlib/soxl/calendars/day.txt",
    source_paths=("provider:research_fixture",),
    start="2026-01-01",
    end="2026-01-05",
    as_of="2026-01-05",
)
registry_path = EXPERIMENT_DIR / "data" / "data_registry.json"
publish_canonical_snapshot(
    snapshot,
    output_dir=EXPERIMENT_DIR / "data" / "snapshots" / "schedule_paper_difference_daily" / "2026-01-05",
    dataset_key="daily.schedule_paper_difference",
    registry_path=registry_path,
    published_at="2026-01-05T21:00:00Z",
)

dashboard_payload = {
    "schema_version": "dashboard.model_matrix.v1",
    "artifact_id": "schedule_paper_execution_model_difference_v1",
    "artifact_type": "dashboard_export",
    "top_models": [
        {
            "rank": 1,
            "model": "random_forest_clf__all_features",
            "base_model": "random_forest_clf",
            "feature_set": "all_features",
            "avg_position": 0.74,
            "stability_score": 54.2,
            "max_drawdown_pct": 21.6,
        }
    ],
    "model_difference_explanations": [
        {
            "schema_version": "analysis.model_difference_explanation.v1",
            "priority_rank": 1,
            "segment": "stress_window::stress",
            "baseline_model": "ridge_mse__all_features",
            "winner_model": "ridge_mse__all_features",
            "loser_model": "random_forest_clf__all_features",
            "relative_return_pct_spread": 6.5,
            "explanation_signal": "baseline_defends",
            "decision_hint": "block_or_rework_loser",
            "review_note": "中文 review：top model 在 stress regime 是 loser，schedule paper execution 必须暴露降仓原因。",
        }
    ],
}
dashboard_path = EXPERIMENT_DIR / "dashboard_export.json"
write_json(dashboard_path, dashboard_payload)

events: list[str] = []
config = PaperExecutionPipelineConfig(
    name="paper_execution_model_difference_v1",
    dashboard_export_path=dashboard_path,
    data_registry_path=registry_path,
    dataset_key="daily.schedule_paper_difference",
    instrument="SOXL",
    intent_ledger_path=EXPERIMENT_DIR / "intent_ledger.json",
    paper_execution_result_path=EXPERIMENT_DIR / "paper_execution_result.json",
    initial_cash=1000.0,
    min_position=0.1,
    max_position=0.75,
    notify_details=True,
)
result = run_paper_execution_once(config, lock_dir=EXPERIMENT_DIR / "locks", notify=events.append)
result.status, events[0]


('success',
 'paper_execution_model_difference_v1:success | mode=paper | gate=blocked_to_min_position | model=random_forest_clf__all_features | target=0.1 | quote=SOXL@2026-01-05/12 | shares=8 | ledger=/Users/rziiiii/dev/workspace/research/schedule_paper_execution_model_difference_v1/intent_ledger.json | model_difference=baseline_defends/block_or_rework_loser stress_window::stress random_forest_clf__all_features->ridge_mse__all_features spread=6.5')

In [2]:
ledger = json.loads(config.intent_ledger_path.read_text(encoding="utf-8"))
execution = json.loads(config.paper_execution_result_path.read_text(encoding="utf-8"))
summary = {
    "status": result.status,
    "schema_version": "research.schedule_paper_execution_model_difference_v1",
    "notification": events[0],
    "source_model": result.payload["source_model"],
    "execution_gate": result.payload["execution_gate"],
    "target_position": result.payload["target_position"],
    "paper_snapshot_shares": result.payload["snapshot"]["shares"],
    "quote_date": result.payload["quote"]["date"],
    "quote_price": result.payload["quote"]["price"],
    "model_difference_schema_version": result.payload["model_difference_schema_version"],
    "model_difference_top_signal": result.payload["model_difference_top_signal"],
    "model_difference_top_decision_hint": result.payload["model_difference_top_decision_hint"],
    "model_difference_top_segment": result.payload["model_difference_top_segment"],
    "model_difference_top_loser_model": result.payload["model_difference_top_loser_model"],
    "model_difference_top_winner_model": result.payload["model_difference_top_winner_model"],
    "model_difference_top_spread": result.payload["model_difference_top_spread"],
    "model_difference_marks_source_loser": result.payload["model_difference_marks_source_loser"],
    "ledger_target_position": ledger["intent"]["target_position"],
    "execution_result_status": execution["status"],
    "sidecars": {
        "dashboard_export": str(dashboard_path),
        "data_registry": str(registry_path),
        "intent_ledger": str(config.intent_ledger_path),
        "paper_execution_result": str(config.paper_execution_result_path),
        "analysis_summary": str(EXPERIMENT_DIR / "analysis_summary.json"),
        "notification_events": str(EXPERIMENT_DIR / "notification_events.json"),
    },
}
write_json(EXPERIMENT_DIR / "notification_events.json", events)
write_json(EXPERIMENT_DIR / "analysis_summary.json", summary)

conclusion = """# 中文结论

- 做了什么：验证 `src/schedule` 的 paper execution pipeline 从 dashboard export 和 canonical data registry 运行到 paper adapter，并在通知、payload、ledger 中保留模型差异解释。
- 核心指标：top model `random_forest_clf__all_features` 原始仓位来自 dashboard 的 `avg_position=0.74`，因 `block_or_rework_loser` 被降到 `target_position=0.1`，paper 结果持有 8 股。
- regime 差异：`stress_window::stress` 中 winner 为 `ridge_mse__all_features`，loser 为 `random_forest_clf__all_features`，spread 为 `6.5`。
- 是否晋升：这是 schedule/API paper/sim 审计链路验证，可随框架进入 production；不是策略晋升。
- 风险：本实验使用最小 fixture 数据和 mock dashboard，不证明真实模型表现。
- 下一步：真实 production notebook 生成 `model_difference_explanations` 后，paper execution 通知会自动暴露该降仓原因。"""

(EXPERIMENT_DIR / "analysis_conclusion.md").write_text(conclusion, encoding="utf-8")
summary


{'status': 'success',
 'schema_version': 'research.schedule_paper_execution_model_difference_v1',
 'notification': 'paper_execution_model_difference_v1:success | mode=paper | gate=blocked_to_min_position | model=random_forest_clf__all_features | target=0.1 | quote=SOXL@2026-01-05/12 | shares=8 | ledger=/Users/rziiiii/dev/workspace/research/schedule_paper_execution_model_difference_v1/intent_ledger.json | model_difference=baseline_defends/block_or_rework_loser stress_window::stress random_forest_clf__all_features->ridge_mse__all_features spread=6.5',
 'source_model': 'random_forest_clf__all_features',
 'execution_gate': 'blocked_to_min_position',
 'target_position': 0.1,
 'paper_snapshot_shares': 8,
 'quote_date': '2026-01-05',
 'quote_price': 12.0,
 'model_difference_schema_version': 'analysis.model_difference_explanation.v1',
 'model_difference_top_signal': 'baseline_defends',
 'model_difference_top_decision_hint': 'block_or_rework_loser',
 'model_difference_top_segment': 'stress_wind

# 中文结论

- 做了什么：验证 `src/schedule` 的 paper execution pipeline 会把 dashboard 中的模型差异解释透传到 schedule payload、详细通知、intent ledger 和 paper execution result。
- 核心指标：top model `random_forest_clf__all_features` 的 `avg_position=0.74`，被 `block_or_rework_loser` 降到 `target_position=0.1`。
- regime 差异：`stress_window::stress` 中 `ridge_mse__all_features` 是 winner，`random_forest_clf__all_features` 是 loser，spread 为 `6.5`。
- 是否晋升：这是框架审计能力验证，可随 schedule/API paper/sim 链路进入 production；不是策略晋升。
- 风险：使用 fixture 数据和 mock dashboard，不证明真实模型收益。
- 下一步：真实 dashboard export 一旦包含 `model_difference_explanations`，paper execution notification 会显示具体降仓原因。